# Verify Module 1: LRLP patch sequence extraction

`fusion_avsr.models.landmark.lrlp.extract_lrlp_sequence` is the module most
exposed to silent correctness bugs (wrong landmark index, off-by-one patch
centering, coordinate-space mismatch) -- none of which would necessarily
throw an error. This notebook verifies it VISUALLY, overlay-style, on real
clips from all three sources (GRID, LRS3-trainval, LRS3-test), the same
approach used by `04_verify_landmark_patches.ipynb`'s Parts 1-3 for raw
landmark/pixel alignment.

**What "looks right" means:**
- `patches.shape == (38, T, 32, 32)`, `raw_coords.shape == (38, T, 2)`.
- Each of the 38 patches, individually, is centered on the right facial
  region: mouth patches show lip texture, jaw patches show the jaw
  contour, nose patches show the nostril row -- not blank/black squares
  (which would indicate a coordinate-space mismatch) and not obviously
  offset from the visible feature (which would indicate an indexing bug).
- The nose-tip-aligned coordinates (`align_to_nose_tip`) put the nose-row
  LRLPs very close to (0, 0), since they sit near the datum point itself.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # adjust if running from somewhere else
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# TODO: fill in correct paths (same convention as notebook 01)
PROJECT_NAME = "your_project_name"
DATASET_ROOT = Path(f"/scratch/{PROJECT_NAME}/datasets")
LRS3_ROOT = DATASET_ROOT / "lrs3"

grid_all_path = "kaggle_lipnet/datasets/jedidiahangekouakou/grid-corpus-dataset-for-training-lipnet/versions/1/data"
GRID_ROOT = DATASET_ROOT / grid_all_path
GRID_LANDMARKS_ROOT = DATASET_ROOT / "grid_landmarks"

LRS3_TRAINVAL_LANDMARKS_ROOT = LRS3_ROOT / "landmarks" / "LRS3_landmarks" / "trainval"
LRS3_TRAINVAL_VIDEO_ROOT = LRS3_ROOT / "ainncy" / "trainval"

LRS3_TEST_VIDEO_ROOT = LRS3_ROOT / "test"
LRS3_TEST_LANDMARKS_ROOT = LRS3_ROOT / "landmarks" / "LRS3_landmarks" / "test"

AUDIO_OUTPUT_DIR = DATASET_ROOT / "extracted_audio"
LIMIT = 5  # a handful of clips per source, not the whole dataset

SEED = 42

In [ ]:
import pickle

import numpy as np
import pandas as pd
import torch
from torchcodec.decoders import VideoDecoder

from fusion_avsr.data.manifest_builder import build_grid_manifest, build_lrs3_manifest, load_or_build_manifest
from fusion_avsr.data.paths import MANIFEST_DIR

# These are cached under MANIFEST_DIR: built once (here or by notebook 01,
# whichever runs first), loaded from the cached CSV every time after.
grid_manifest = load_or_build_manifest(
    MANIFEST_DIR / "grid_manifest.csv", build_grid_manifest,
    grid_root=GRID_ROOT, landmarks_root=GRID_LANDMARKS_ROOT,
    audio_output_dir=AUDIO_OUTPUT_DIR, limit=LIMIT,
)
lrs3_trainval_manifest = load_or_build_manifest(
    MANIFEST_DIR / "lrs3_trainval_manifest.csv", build_lrs3_manifest,
    video_root=LRS3_TRAINVAL_VIDEO_ROOT, audio_output_dir=AUDIO_OUTPUT_DIR,
    landmarks_root=LRS3_TRAINVAL_LANDMARKS_ROOT, source="lrs3_trainval", limit=LIMIT,
)
lrs3_test_manifest = load_or_build_manifest(
    MANIFEST_DIR / "lrs3_test_manifest.csv", build_lrs3_manifest,
    video_root=LRS3_TEST_VIDEO_ROOT, audio_output_dir=AUDIO_OUTPUT_DIR,
    landmarks_root=LRS3_TEST_LANDMARKS_ROOT, source="lrs3_test", limit=LIMIT,
)

MANIFESTS = {
    "grid": grid_manifest,
    "lrs3_trainval": lrs3_trainval_manifest,
    "lrs3_test": lrs3_test_manifest,
}
for name, m in MANIFESTS.items():
    print(name, m.shape)

In [ ]:
from fusion_avsr.models.landmark.lrlp import (
    LRLP_INDICES,
    NOSE_TIP_INDEX,
    NUM_LRLPS,
    align_to_nose_tip,
    extract_lrlp_sequence,
)

print(f"NUM_LRLPS={NUM_LRLPS}, NOSE_TIP_INDEX={NOSE_TIP_INDEX}")

In [ ]:
def load_one_clip(manifest, seed=SEED):
    """Pick one random row and decode its frames + landmarks."""
    row = manifest.sample(n=1, random_state=seed).iloc[0]
    decoder = VideoDecoder(row["video_path"], dimension_order="NHWC")
    frames = np.stack([decoder[t].numpy() for t in range(len(decoder))])
    with open(row["landmark_path"], "rb") as f:
        landmarks = pickle.load(f)
    n = min(len(frames), len(landmarks))
    return row["sample_id"], frames[:n], landmarks[:n]

## Run Module 1 on one clip from each source

In [ ]:
import matplotlib.pyplot as plt

results = {}
for source_name, manifest in MANIFESTS.items():
    sample_id, frames, landmarks = load_one_clip(manifest)
    patches, raw_coords, valid_mask = extract_lrlp_sequence(frames, landmarks)
    aligned_coords = align_to_nose_tip(raw_coords, landmarks)
    results[source_name] = (sample_id, frames, landmarks, patches, raw_coords, aligned_coords, valid_mask)
    print(
        f"{source_name} ({sample_id}): patches={patches.shape}, "
        f"raw_coords={raw_coords.shape}, valid_frames={valid_mask.sum()}/{len(valid_mask)}"
    )

## Overlay: original frame with the 38 LRLPs highlighted (mouth/jaw/nose colored separately)

In [ ]:
MOUTH_SLICE = slice(0, 20)   # LRLP_INDICES[0:20]  = mouth 48-67
JAW_SLICE = slice(20, 33)    # LRLP_INDICES[20:33] = jaw 2-14
NOSE_SLICE = slice(33, 38)   # LRLP_INDICES[33:38] = nose 31-35

fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 6))
for ax, (source_name, (sample_id, frames, landmarks, patches, raw_coords, aligned_coords, valid_mask)) in zip(axes, results.items()):
    frame_index = int(np.argmax(valid_mask))  # first valid frame
    ax.imshow(frames[frame_index])
    coords_this_frame = raw_coords[:, frame_index, :]
    ax.scatter(*coords_this_frame[MOUTH_SLICE].T, s=15, c="orange", label="mouth")
    ax.scatter(*coords_this_frame[JAW_SLICE].T, s=15, c="lime", label="jaw")
    ax.scatter(*coords_this_frame[NOSE_SLICE].T, s=15, c="cyan", label="nose")
    ax.set_title(f"{source_name}: {sample_id} (frame {frame_index})")
    ax.legend(loc="upper right", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Mosaic: all 38 extracted 32x32 patches for one frame, per source

In [ ]:
for source_name, (sample_id, frames, landmarks, patches, raw_coords, aligned_coords, valid_mask) in results.items():
    frame_index = int(np.argmax(valid_mask))
    fig, axes = plt.subplots(4, 10, figsize=(14, 6))
    fig.suptitle(f"{source_name}: {sample_id}, frame {frame_index} -- 38 LRLP patches (mouth=orange border, jaw=lime, nose=cyan)")
    for k, ax in enumerate(axes.ravel()):
        if k < NUM_LRLPS:
            ax.imshow(patches[k, frame_index], cmap="gray")
            border_color = "orange" if k < 20 else ("lime" if k < 33 else "cyan")
            for spine in ax.spines.values():
                spine.set_edgecolor(border_color)
                spine.set_linewidth(2)
            ax.set_xticks([]); ax.set_yticks([])
        else:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

## Nose-tip alignment sanity check

In [ ]:
for source_name, (sample_id, frames, landmarks, patches, raw_coords, aligned_coords, valid_mask) in results.items():
    frame_index = int(np.argmax(valid_mask))
    nose_row_aligned = aligned_coords[33:38, frame_index]  # the 5 nose LRLPs
    print(f"{source_name}: nose-row aligned coords (should be small, near the origin):")
    print(nose_row_aligned)

## Checklist

- [ ] `patches.shape == (38, T, 32, 32)` and `raw_coords.shape == (38, T, 2)` for all three sources.
- [ ] In the overlay plots: orange (mouth) points sit on the lips, lime (jaw) points trace the jawline, cyan (nose) points sit on the nostril row -- for GRID, LRS3-trainval, AND LRS3-test.
- [ ] In the patch mosaics: mouth patches visibly show lip/mouth texture, jaw patches show jaw/cheek skin, nose patches show the nostril area -- none are blank/black (which would mean the coordinate fell outside the frame) or show the wrong part of the face.
- [ ] Nose-row aligned coordinates are close to (0, 0) (a few pixels at most), confirming `align_to_nose_tip` is subtracting the right reference point.
- [ ] Any clip with a `valid_frames` count well below the total frame count is noted (frequent missed detections would mean a lot of forward/back-filled frames).